Authors: Myles Lewis & Joonseo Bang

Background information:
This dataset represents diabetes patient records and these were obtained from two sources. An automatic electronic recording device and paper records. In this notebook we will explore the dataset, organize the patient information, and use a Support Vector Machine (SVM) model to analyze the data and make predictions.

Dataset: https://archive.ics.uci.edu/dataset/34/diabetes


In [ ]:
import os
import tarfile
import subprocess
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

In [ ]:
archive = "diabetes-data.tar[1].Z"
if os.path.exists(archive):
    subprocess.run(["uncompress", "-f", archive], check=True)

with tarfile.open("diabetes-data.tar[1]", "r") as tar:
    tar.extractall(".")

/tmp/ipykernel_3396/3935576537.py:6: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(".")


Had trouble with extracting dataset prior to the rest of the code. So this cell helps python read it.

In [ ]:
records = []
for patient_id in range(1, 71):
    filename = f"Diabetes-Data/data-{patient_id:02d}"
    patient = pd.read_csv(filename, sep="\t", header=None,
                          names=["Date", "Time", "Code", "Value"])
    patient["Patient_ID"] = patient_id
    records.append(patient)

df = pd.concat(records, ignore_index=True)
df.head()

,Date,Time,Code,Value,Patient_ID
0,04-21-1991,9:09,58,100,1
1,04-21-1991,9:09,33,9,1
2,04-21-1991,9:09,34,13,1
3,04-21-1991,17:08,62,119,1
4,04-21-1991,17:08,33,7,1


Combines the 70 patient files into one table. Each record contains a date, time, event code, and value.

In [ ]:
glucose_codes = [48, 57, 58, 59, 60, 61, 62, 63, 64]
df = df[df["Code"].isin(glucose_codes)].copy()
df["Value"] = pd.to_numeric(df["Value"], errors="coerce")
df["Hour"] = pd.to_datetime(df["Time"], format="%H:%M", errors="coerce").dt.hour
df = df.dropna(subset=["Hour", "Value"])
df["High_Glucose"] = (df["Value"] >= 200).astype(int)
df.head()

,Date,Time,Code,Value,Patient_ID,Hour,High_Glucose
0,04-21-1991,9:09,58,100.0,1,9.0,0
3,04-21-1991,17:08,62,119.0,1,17.0,0
5,04-21-1991,22:51,48,123.0,1,22.0,0
6,04-22-1991,7:35,58,216.0,1,7.0,1
10,04-22-1991,16:56,62,211.0,1,16.0,1


Keeps only blood glucose readings and labels readings of 200 or higher as high (1); other readings are labeled 0.

In [ ]:
X = df[["Patient_ID", "Hour", "Code"]]
y = df["High_Glucose"]

Uses patient number, hour, and measurement code as inputs.

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

Scales the input features so they are on a similar range.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Uses 80% of the records to train the model and 20% to test it.

In [ ]:
model = SVC()

Creates a Support Vector Machine classifier.

In [ ]:
model.fit(X_train, y_train)

SVC()

Fits the SVM model using the training data.

In [ ]:
model_predict = model.predict(X_test)

Predicts whether each test record has a high glucose reading.

In [ ]:
accuracy = accuracy_score(y_test, model_predict)
print("Accuracy:", accuracy)

Accuracy: 0.7194085027726432


Shows the percentage of test predictions that were correct.

In [ ]:
print(classification_report(y_test, model_predict, zero_division=0))
print("Precision:", precision_score(y_test, model_predict, zero_division=0))
print("Recall:", recall_score(y_test, model_predict, zero_division=0))
print("F1-Score:", f1_score(y_test, model_predict, zero_division=0))

              precision    recall  f1-score   support

           0       0.72      0.99      0.83      1935
           1       0.62      0.04      0.07       770

    accuracy                           0.72      2705
   macro avg       0.67      0.51      0.45      2705
weighted avg       0.69      0.72      0.62      2705

Precision: 0.6170212765957447
Recall: 0.03766233766233766
F1-Score: 0.07099143206854346


Reports precision, recall, and F1-score to show how well the model identifies high-glucose readings.

Conclusion: The SVM model achieved an accuracy of approximately 71.9% on the test data. However, it had difficulty identifying high glucose readings, with a recall of only 3.8%. This shows that although the model correctly predicted many normal readings, it was not very effective at detecting high glucose levels. Overall, this exercise demonstrated how machine learning can be used to analyze diabetes patient records and the importance of evaluating model performance beyond accuracy alone.